In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("popular_anime.csv")
print("Dataset shape :", df.shape)
print("Columns       :", list(df.columns)[:9])

In [ ]:
# ---------- CRISP-DM Phase 1 : Business Understanding ----------
objective = ("Identify what drives the user rating of an anime title, "
             "so that popular titles can be recommended reliably.")
print("Objective :", objective)

# ---------- CRISP-DM Phase 2 : Data Understanding ----------
print("\nRecords :", len(df))
print("Missing scores :", df["score"].isna().sum())
print("\nScore summary :")
print(df["score"].describe().round(3))

In [ ]:
# ---------- CRISP-DM Phase 3 : Data Preparation ----------
crisp = df[["name", "type", "episodes", "score", "scored_by", "rank"]].dropna()
crisp = crisp[crisp["episodes"] < 200]

print("Prepared records :", len(crisp))
print(crisp.head())

In [ ]:
# ---------- CRISP-DM Phase 4 : Modeling ----------
x = np.log10(crisp["scored_by"])
y = crisp["score"]
slope, intercept = np.polyfit(x, y, 1)
print("Model : score = %.4f * log10(scored_by) + %.4f" % (slope, intercept))

# ---------- CRISP-DM Phase 5 : Evaluation ----------
pred = slope * x + intercept
r2 = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()
print("R-squared :", round(r2, 4))
print("Mean absolute error :", round(np.abs(y - pred).mean(), 4))

In [ ]:
# ---------- CRISP-DM Phase 6 : Deployment ----------
def predict_score(votes):
    """Reusable function that deploys the fitted model."""
    return round(slope * np.log10(votes) + intercept, 2)

for v in [500, 5000, 50000, 500000]:
    print("Votes =", v, "-> predicted score =", predict_score(v))

In [ ]:
# ---------- SEMMA Step 1 : Sample ----------
sample = df.dropna(subset=["score"]).sample(n=2000, random_state=42)
print("Sample size :", len(sample))

# ---------- SEMMA Step 2 : Explore ----------
print("\nMean score in sample :", round(sample["score"].mean(), 3))
print("Mean score in full    :", round(df["score"].mean(), 3))
print("\nCounts by type :")
print(sample["type"].value_counts().head())

In [ ]:
# ---------- SEMMA Step 3 : Modify ----------
sample = sample.copy()
sample["popularity"] = np.log10(sample["scored_by"])
sample["is_series"] = (sample["type"] == "TV").astype(int)
print(sample[["name", "score", "popularity", "is_series"]].head(3))

# ---------- SEMMA Step 4 : Model ----------
m, c = np.polyfit(sample["popularity"], sample["score"], 1)
sample["predicted"] = m * sample["popularity"] + c

# ---------- SEMMA Step 5 : Assess ----------
rmse = np.sqrt(((sample["score"] - sample["predicted"]) ** 2).mean())
print("\nRMSE on sample :", round(rmse, 4))

In [ ]:
# ---------- KDD Step 1 : Selection ----------
kdd = df[["name", "genres", "type", "score", "scored_by"]]
print("Selected columns :", list(kdd.columns))

# ---------- KDD Step 2 : Preprocessing ----------
kdd = kdd.dropna(subset=["genres", "score"])
print("Records after cleaning :", len(kdd))

In [ ]:
# ---------- KDD Step 3 : Transformation ----------
kdd = kdd.copy()
kdd["genre_list"] = kdd["genres"].str.split(", ")
exploded = kdd.explode("genre_list")
print("Rows after exploding genres :", len(exploded))

# ---------- KDD Step 4 : Data Mining ----------
pattern = (exploded.groupby("genre_list")["score"]
           .agg(["count", "mean"])
           .query("count >= 200")
           .sort_values("mean", ascending=False))
print("\nTop rated genres :")
print(pattern.head(5).round(3))

In [ ]:
# ---------- KDD Step 5 : Interpretation and Evaluation ----------
best = pattern.index[0]
print("Knowledge discovered :")
print("The", best, "genre carries the highest average rating of",
      round(pattern.iloc[0]["mean"], 2), "across", int(pattern.iloc[0]["count"]), "titles.")

# ---------- Comparison of the three frameworks ----------
comparison = pd.DataFrame({
    "Framework": ["CRISP-DM", "SEMMA", "KDD"],
    "Phases": [6, 5, 5],
    "Focus": ["Business driven, iterative",
              "Sampling and modelling driven",
              "Knowledge discovery driven"]
})
print()
print(comparison.to_string(index=False))